In [ ]:
#Install requirements
%pip install -r "../Requirements.txt"

In [ ]:
#Import libraries
import pandas as pd 
import os
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
import scanpy as sc
from scipy.sparse import issparse, hstack, csr_matrix
import sys


In [ ]:
#Add the src to the path
sys.path.append(os.path.abspath(os.path.join('..')))

In [ ]:
#Define paths for original data and output directories
data_dir = Path('../data/GSE107451_DGRP-551_w1118_WholeBrain_57k_0d_1d_3d_6d_9d_15d_30d_50d_10X_DGEM_MEX.mtx.tsv') #folder with the matrix, genes.tsv, barcodes.tsv
annot_path = Path('../data/GSE107451_DGRP-551_w1118_WholeBrain_57k_0d_1d_3d_6d_9d_15d_30d_50d_10X_DGEM_MEX.mtx.tsv/annotation.tsv') #annotation file
metadata = Path('../data/GSE107451_DGRP-551_w1118_WholeBrain_57k_Metadata.tsv/GSE107451_DGRP-551_w1118_WholeBrain_57k_Metadata.tsv') #metadata

output_dir = Path('../data/data_sets')
output_dir.mkdir(exist_ok=True)

output_dir_figs = Path('../figures/data_preprocessing')
output_dir_figs.mkdir(exist_ok=True)

In [ ]:
#Load the Matrix
adata = sc.read_10x_mtx(data_dir, var_names="gene_symbols", cache=False)

print(f"The matrix loaded successfully. It consists of {adata.n_obs} cells x {adata.n_vars} genes")

In [ ]:
#Load metadata and merge some columns
meta = pd.read_csv(metadata, sep = "\t") #cell annotation, QC metrics, Genotype, sex, age
barcode_col = "new_barcode"
meta = meta.set_index(barcode_col) #set the index of the metadata file to be the new_barcode column

#columns from the metadata file that we need
meta_cols_to_keep = [
    "annotation", # cell type label
    "Age", #timepoints in days
    "Replicate", #true biological replicates
    "Genotype", #line
    "nGene",  # number of genes detected
    "nUMI",  # total UMI counts
    "percent.mito", # mitochondrial gene percentage
    "sex"
]

#check if the columns exist
meta_cols_to_keep = [c for c in meta_cols_to_keep if c in meta.columns]
meta = meta[meta_cols_to_keep]

#Merge the information from the metadata into the adata objects
adata.obs = adata.obs.join(meta, how="left") #.join() joins on the index by default

In [ ]:
print(f"adata.obs columns {list(adata.obs.columns)}")
print(f"Unique lines: {adata.obs['Genotype'].unique()}")
print(f"Cells missing cell type label:  {adata.obs['annotation'].isna().sum():,}")
print(f"Unique time points: {sorted(adata.obs['Age'].unique())}")
print(f"Unique cell types: {adata.obs['annotation'].nunique()}")

In [ ]:
#Selected timepoints.
timepoints = [0, 1, 3, 6, 9,15,30,50] #very young, young, mid-young, mid-aging, aging. 50 days is missing in one genotype

age_map = {
    0: "eclosion",
    1: "1_very_young",
    3: "2_very_young",
    6: "1_young",
    9: "2_young",
    15: "mid_young",
    30: "mid_aged",
    50: "aged"
}

#Filter for the selected timepoints
adata = adata[adata.obs["Age"].isin(timepoints)].copy()

adata.obs["age_map"] = adata.obs["Age"].map(age_map)

print(adata.obs.groupby(["Genotype", "Age"]).size().unstack(fill_value=0).to_string())

In [ ]:
age_map_order = ["eclosion", "1_very_young", "2_very_young", "1_young", "2_young", "mid_young", "mid_aged", "aged"]
age_map_colors = ["#2C3E7A",  "#3A9BD5", "#5CB85C",  "#A8D44A",  "#F0AD4E", "#E8733A",  "#C0392B", "#7B1E1E"]

In [ ]:
#Remove cell clusters with unknown/non-biologically meaningfull label (numeric labels)
from src.preprocessing_functions import is_named

named_mask = adata.obs["annotation"].apply(is_named)
removed_clusters = sorted(
    adata.obs.loc[~named_mask, "annotation"].dropna().unique(), key=lambda x: int(x) if str(x).strip().isdigit() else 0)

print(f"Removed clusters: {removed_clusters}")

adata = adata[named_mask].copy()
print(f"{adata.obs['annotation'].nunique()} named cell types")

In [ ]:
#Remove Hsp cluster - maybe some cells were clustered together due to the activation of Hsp proteins due to stress. This cluster is an outlier.

non_biological = ["Hsp"]
adata = adata[~adata.obs["annotation"].isin(non_biological)].copy()
print(f"Removed non-biological clusters")
print(f"Remaining cells: {adata.obs['annotation'].nunique()} clusters")

In [ ]:
#Normalize expression with the same way in both pipelines
adata.raw = adata
adata.layers["raw_counts"] = adata.X.copy()

In [ ]:
#Visualizations before preprocessing steps

n_sample = min(3000, adata.n_obs)
idx_sample = np.random.choice(adata.n_obs, n_sample, replace=False) #select some of the cells (not all 57,000) just for visualization
X_sample = adata.X[idx_sample]

if issparse(X_sample):
    X_sample = X_sample.toarray()

raw_nonzero = X_sample[X_sample >0].flatten() #visualize only the nonzero expression genes at this step

fig, axes = plt.subplots(1,3, figsize=(15,4))

#Distribution of raw counts - we expect right-skewed distribution
axes[0].hist(raw_nonzero, bins=100, color="#2E75B6", edgecolor="none", log=True)
axes[0].set_title("Raw UMI counts")
axes[0].set_xlabel("UMI count")
axes[0].set_ylabel("Frequency (log)")

#Total UMIs per cell
axes[1].hist(adata.obs["nUMI"].dropna(), bins=80, color='#E8733A', edgecolor="none")
axes[1].set_title("Total UMIs per cell")
axes[1].set_xlabel("Total UMI count")
axes[1].set_ylabel("Number of cells")

#Fraction of zeros per cell
zero_frac = (X_sample ==0).mean(axis=1)
axes[2].hist(zero_frac, bins=60, color="#5CB85C", edgecolor="none")
axes[2].axvline(zero_frac.mean(), color="red", linestyle="--", label=f"Mean: {zero_frac.mean(): .1%}")
axes[2].set_title("Fraction of zeros per cell")
axes[2].set_xlabel("Fraction of genes = 0")
axes[2].set_ylabel("Number of cells")
axes[2].legend()


plt.suptitle("Expression matrix before normalization", fontsize=13)
plt.tight_layout()
plt.savefig(output_dir_figs / "before_normalization.png", dpi=300)
plt.show()

In [ ]:
#Scale every cell to 10,000 total counts
sc.pp.normalize_total(adata, target_sum=1e4)
#Log transform - makes the distribution more normal
sc.pp.log1p(adata)
print(f" normalize_total(10k) + log1p applied")

adata.layers["normalised"] = adata.X.copy()

In [ ]:
#Visualizations  after normalization

X_norm_sample = adata.X[idx_sample]

if issparse(X_norm_sample):
    X_norm_sample = X_norm_sample.toarray()

norm_nonzero = X_norm_sample[X_norm_sample >0].flatten() #visualize only the nonzero expression genes 

fig, axes = plt.subplots(1,3, figsize=(16,4))

#Distribution of raw counts before
axes[0].hist(raw_nonzero, bins=100, color="#AAAAAA", edgecolor="none", log=True)
axes[0].set_title("Raw UMI counts")
axes[0].set_xlabel("UMI count")
axes[0].set_ylabel("Frequency (log)")

#after normalization
axes[1].hist(norm_nonzero, bins=100, color="#2E75B6", edgecolor="none")
axes[1].set_title("After normalization + log1p")
axes[1].set_xlabel("log1p expression value")
axes[1].set_ylabel("Frequency")
axes[1].set_xlim(0, 12) 

#Gene mean vs variance
gene_mean = np.array(adata.X.mean(axis=0)).flatten()
if issparse(adata.X):
    gene_var = np.array(adata.X.power(2).mean(axis=0)).flatten() - gene_mean**2
else:
    gene_var = np.var(adata.X, axis=0)
 
n_genes = len(gene_mean)
plot_idx = np.random.choice(n_genes, min(5000, n_genes), replace=False)
axes[2].scatter(gene_mean[plot_idx], gene_var[plot_idx],
                alpha=0.2, s=2, color="#2E75B6")
axes[2].set_xscale("log")
axes[2].set_yscale("log")
axes[2].set_title("Mean vs variance per gene (after normalization)")
axes[2].set_xlabel("Mean expression (log)")
axes[2].set_ylabel("Variance (log)")


plt.suptitle("Expression matrix after normalization", fontsize=13)
plt.tight_layout()
plt.savefig(output_dir_figs / "after_normalization.png", dpi=300)
plt.show()

In [ ]:
#Define functional family groups
cell_type_groups = {
    #KENYON CELLS
    "G-KC": "Kenyon_cells",
    "A/B-KC": "Kenyon_cells",
    "A/B*-KC": "Kenyon_cells",

    #OPTIC LOBE NEURONS
    "T1": "Optic_lobe", "T2": "Optic_lobe", "T3": "Optic_lobe",
    "T4/T5": "Optic_lobe",
    "Mi1": "Optic_lobe",
    "Tm1/TmY8": "Optic_lobe", "Tm5ab": "Optic_lobe", "Tm5c": "Optic_lobe",
    "Tm9": "Optic_lobe", "TmY14": "Optic_lobe",
    "Dm8/Dm11": "Optic_lobe", "Dm9": "Optic_lobe",
    "L1": "Optic_lobe", "L2": "Optic_lobe", "L3": "Optic_lobe", "L4/L5": "Optic_lobe",
    "Lamina_monopolar": "Optic_lobe",
    "Lawf1": "Optic_lobe", "Lawf2": "Optic_lobe",
    "C2": "Optic_lobe", "C3": "Optic_lobe",
    "Pm1/Pm2": "Optic_lobe", "Pm1/Pm2/Pm3": "Optic_lobe",
    "Pm3": "Optic_lobe", "Pm4": "Optic_lobe",
    "Photoreceptors": "Optic_lobe",
    "Proc/Gpb5": "Optic_lobe", "Proc/Ms": "Optic_lobe",

    #MONOAMINERGIC NEURONS
    "Dopaminergic": "Monoaminergic",
    "Serotonergic": "Monoaminergic",
    "Tyraminergic": "Monoaminergic",
    "Octopaminergic": "Monoaminergic",
    "PAM": "Monoaminergic",

    #GLIA
    "Ensheathing_glia": "Glia",
    "Astrocyte-like": "Glia",
    "Cortex_glia": "Glia",
    "Perineurial_glia": "Glia",
    "Subperineurial_glia": "Glia",
    "Chiasm_glia": "Glia",

    #OLFACTORY PROJECTION NEURONS
    "adPN": "Olfactory_PN",
    "adPN/C15": "Olfactory_PN",
    "adPN/C15&kn": "Olfactory_PN",
    "adPN/kn": "Olfactory_PN",
    "adPN/kn&CG31676": "Olfactory_PN",
    "Olfactory_projection_neurons": "Olfactory_PN",

    #PEPTIDERGIC / NEUROSECRETORY NEURONS
    "Peptidergic": "Peptidergic",
    "CCAP": "Peptidergic",
    "CCHa1": "Peptidergic",
    "Capa": "Peptidergic",
    "Crz": "Peptidergic",
    "Hug": "Peptidergic",
    "ITP": "Peptidergic",
    "Mip": "Peptidergic", "Mip/ITP": "Peptidergic", "Mip/OCT": "Peptidergic",
    "FMRFa": "Peptidergic",
    "AstA/NPF": "Peptidergic", "AstA/Nplp1": "Peptidergic",
    "IPC": "Peptidergic",   

    #CLOCK/CIRCADIAN NEURON
    "Clock": "Clock",
    "DN1": "Clock", "DCN": "Clock",
    "LNv": "Clock",

    #IMMUNE CELLS
    "Plasmatocytes": "Immune",
}

adata.obs["cell_group"] = adata.obs["annotation"].map(cell_type_groups)

In [ ]:
#Optional analysis - neuro vs glia
glia_clusters = ["Ensheathing_glia", "Astrocyte-like", "Cortex_glia", "Perineurial_glia", "Subperineurial_glia", "Chiasm_glia"]
adata.obs["neuro_glia"] = np.where(adata.obs["annotation"].isin(glia_clusters), "Glia", "Neuron")

In [ ]:
mapped = adata.obs["cell_group"].notna().sum()
unmapped = adata.obs["cell_group"].isna().sum()

print(f"Cells mapped to family: {mapped:,}")
print(f"Cells not mapped: {unmapped:,}")
print(adata.obs.loc[adata.obs['cell_group'].isna(), 'annotation'].value_counts().to_string())
print(adata.obs["cell_group"].value_counts().to_string())

In [ ]:
#Filter out fine cell types with 1 cell in any timepoint

ct_age = (adata.obs
          .groupby(["annotation", "age_map"])
          .size()
          .unstack(fill_value=0))

from src.preprocessing_functions import passes_filter
valid_celltypes = ct_age[ct_age.apply(passes_filter, axis=1)].index.tolist()
removed_celltypes = [ct for ct in ct_age.index if ct not in valid_celltypes]

# Keep only cells belonging to valid cell types
before = adata.n_obs
adata = adata[adata.obs["annotation"].isin(valid_celltypes)].copy()

print(f"Remaining: {adata.n_obs:,} cells, {adata.obs['annotation'].nunique()} cell types")

In [ ]:
#Group surviving fine clusters to families 
adata.obs["cell_group"] = adata.obs["annotation"].map(cell_type_groups)

In [ ]:
min_cells_per_bin = 20 #minimum cells required in every age timepoint

group_summary = []
pass_groups = []

for group in sorted(adata.obs["cell_group"].dropna().unique()):
    sub = adata.obs[adata.obs["cell_group"] == group]
    bin_counts = sub["age_map"].value_counts().reindex(age_map_order, fill_value=0)
    n_bins_present = (bin_counts > 0).sum()
    weakest_bin = bin_counts.idxmin()
    weakest_count = int(bin_counts.min())
    pass_class = bool((bin_counts >= min_cells_per_bin).all())

    if pass_class:
        pass_groups.append(group)

    print(f"\n{group} : {len(sub):,} cells | bins present: {n_bins_present}")

    group_summary.append({
        "group": group,
        "total_cells": len(sub),
        "n_fine_clusters": sub["annotation"].nunique(),
        "weakest_bin": weakest_bin,
        "weakest_count": weakest_count,
        "classification_ready": pass_class
    })

summary_df = pd.DataFrame(group_summary)
summary_df.to_csv( output_dir/ "group_summary.csv", index=False)
print(summary_df.to_string(index=False))

In [ ]:
#Genertae one dataset per group

for group in pass_groups:
    group_adata = adata[adata.obs["cell_group"] == group].copy()
    safe_name = group.replace("/", "_")
    out_path = output_dir / f"{safe_name}.h5ad"
    group_adata.write_h5ad(out_path)

#Save the full annotated dataset
adata.write_h5ad(output_dir / "full_dataset.h5ad")

#Optinally save the neuron-glia datasets
for ng in ["Neuron", "Glia"]:
    ng_adata= adata[adata.obs["neuro_glia"] == ng].copy()
    ng_adata.write_h5ad(output_dir / f"NG_{ng}.h5ad")

print(f"All datasets were saved")